In [ ]:
#| default_exp write

## Writing and changing

Cell creation, targeted updates, documentation insertion, and examples.

Writing notebooks safely is harder than appending text to a file. A notebook edit needs to preserve cell ids, clear stale outputs, validate Python when possible, export through nbdev automatically, and avoid overwriting the wrong cell.

This notebook provides the public write path: append or insert cells with `write_nb`, surgically change one cell with `update_cell`, and use file/stdin inputs when multiline shell quoting would be fragile.


There are two writing modes because notebook edits have two different shapes. Use cell-block writes when adding or replacing structured cells, and use literal replacement only for exact renames across existing cells. Both paths stamp nbskill metadata and export affected notebooks automatically when they have an nbdev export target.


In [ ]:
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
from pathlib import Path as _Path
from fastcore.nbio import read_nb as _read_nb
from nbskill.read import nb_overview as _example_nb_overview
from nbskill.write import write_nb as _example_write_nb
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook
from nbskill.write import _export_notebook, batch_edit_nb, update_cell, write_nb

In [ ]:
path = demo_path("02_write_example.ipynb")
try:
    _example_write_nb(str(path), "%%markdown\n## Result\n---\n%%code\nanswer = 42", replace=True)
    _example_nb_overview(str(path), include_docs=True)
finally:
    remove_demo_path(path)

In [ ]:
#| export
import ast
import builtins
import copy
import difflib
import glob
import json
import re
from pathlib import Path

from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import read_nb as _read_nb
from fastcore.nbio import write_nb as _write_nb
from fastcore.script import Param, call_parse, _in_call_parse
from nbdev.doclinks import nbdev_export

from nbskill.execute import run_notebook_test
from nbskill.review import style_check
from nbskill.foundation import (
    cell_source, clear_outputs, cli_error,
    cli_return, find_cell_by_id, find_cell_by_text, load_cells_text,
    exported_py_path, one_chapter, parse_cells, parse_one_cell, replace_cell,
    stamp_export_metadata, stamp_notebook_metadata, tracked_call,
    validate_code_cells,
)
from nbskill.parallel import notebook_locks

### Adding cells

`write_nb` is the broad insertion tool. It can append, insert before or after a stable cell id, replace a whole notebook, or write into a named chapter while preserving notebook structure.

In [ ]:
#| export
def _literal_replacement_mode(old_str, new_str):
    return old_str is not None or new_str is not None


def _resolve_notebook_paths(path):
    raw = str(path)
    pth = Path(raw).expanduser()
    if any(char in raw for char in "*?[]"):
        candidates = [Path(item) for item in glob.glob(raw, recursive=True)]
    elif pth.is_dir():
        candidates = list(pth.rglob("*.ipynb"))
    elif pth.is_file():
        candidates = [pth]
    else:
        candidates = []
    paths = sorted({candidate for candidate in candidates if candidate.suffix == ".ipynb" and ".ipynb_checkpoints" not in candidate.parts})
    if not paths: cli_error(f"No notebooks matched {path!r}")
    return paths


def _literal_cell_diff(before, after, limit=24):
    lines = list(difflib.unified_diff(
        before.splitlines(), after.splitlines(), fromfile="before", tofile="after", lineterm="", n=2,
    ))
    if len(lines) > limit: lines = [*lines[:limit], "... diff truncated ..."]
    return "\n".join(lines)


def _export_notebook(nb, nb_path):
    py_path = exported_py_path(nb_path, nb)
    if py_path is None: return None
    nbdev_export(path=str(nb_path))
    if py_path.exists():
        stamp_export_metadata(nb, py_path)
        _write_nb(nb, nb_path)
    return py_path


def _replace_literal_in_notebook(nb, old_str, new_str, validate_code=True, collect_details=False):
    changed_cells, matches, details = 0, 0, []
    for cell in nb.cells:
        before = cell_source(cell)
        count = before.count(old_str)
        if not count: continue
        after = before.replace(old_str, new_str)
        if validate_code and getattr(cell, "cell_type", None) == "code": validate_code_cells([mk_cell(after)])
        if collect_details:
            details.append({
                "cell_id": getattr(cell, "id", ""),
                "cell_type": getattr(cell, "cell_type", ""),
                "matches": count,
                "diff": _literal_cell_diff(before, after),
            })
        cell.source = after
        clear_outputs(cell)
        changed_cells += 1
        matches += count
    if matches: stamp_notebook_metadata(nb)
    return changed_cells, matches, details


def _format_literal_replacement_details(changed):
    lines = ["Changed cells:"]
    for nb_path, _, _, details in changed:
        for detail in details:
            lines.append(f"- {nb_path} id={detail['cell_id']} type={detail['cell_type']} matches={detail['matches']}")
            if detail["diff"]:
                lines.extend(f"    {line}" for line in detail["diff"].splitlines())
    return "\n".join(lines)


def _write_literal_replacements(path, old_str, new_str, run_test=False, validate_code=True, dry_run=False, show_cells=False):
    if old_str in {None, ""}: cli_error("Pass a non-empty old_str for literal replacements")
    if new_str is None: cli_error("Pass new_str for literal replacements")
    paths = _resolve_notebook_paths(path)
    changed = []
    exported = False
    with notebook_locks(*paths):
        for nb_path in paths:
            nb = _read_nb(nb_path)
            cells_changed, matches, details = _replace_literal_in_notebook(
                nb, old_str, new_str, validate_code=validate_code, collect_details=show_cells,
            )
            if not matches: continue
            changed.append((nb_path, cells_changed, matches, details))
            if not dry_run:
                _write_nb(nb, nb_path)
                exported = _export_notebook(nb, nb_path) is not None or exported
                if run_test: run_notebook_test(nb_path)
    total_matches = sum(matches for _, _, matches, _ in changed)
    total_cells = sum(cells for _, cells, _, _ in changed)
    if not changed:
        msg = f"No matches for {old_str!r} in {len(paths)} notebook(s)"
    else:
        prefix = "Dry run: would replace" if dry_run else "Replaced"
        msg = f"{prefix} {total_matches} matches in {total_cells} cells across {len(changed)} notebook(s)"
        details = "; ".join(f"{nb_path}: {matches} matches/{cells} cells" for nb_path, cells, matches, _ in changed)
        msg += f" ({details})"
        if exported: msg += " and exported with nbdev"
        if show_cells: msg += f"\n{_format_literal_replacement_details(changed)}"
    print(msg)
    return cli_return([path for path, _, _, _ in changed])


@call_parse
@tracked_call
def write_nb(
    path: str,  # Notebook path, directory, or glob when replacing literals
    cells: Param("Cell block text", str, opt=False, nargs="?") = "",  # Cells to write; use - to read stdin
    cells_file: str | None = None,  # Read cell block text from a UTF-8 file to avoid shell escaping
    before_id: str | None = None,  # Insert before this stable cell id
    after_id: str | None = None,  # Insert after this stable cell id
    chapter: str | None = None,  # Chapter title string or regex; missing chapters are created
    replace: bool = False,  # Replace the full notebook, or the selected chapter body
    cell_type: str = "code",  # Default type for cells without %% marker
    run_test: bool = False,  # Execute the notebook with execnb after writing
    run_style: bool = False,  # Run chstyle after writing
    style_strict: bool = False,  # Fail when chstyle finds hints
    validate_code: bool = True,  # Validate new Python code cells before writing
    old_str: str | None = None,  # Literal text to replace across notebook cell sources
    new_str: str | None = None,  # Literal replacement text for old_str
    dry_run: bool = False,  # Show literal replacement plan without writing
    show_cells: bool = False,  # Include touched cell ids and compact diffs for literal replacements
):
    "Write cells to a notebook, or replace literal text across notebooks."
    if _literal_replacement_mode(old_str, new_str):
        if cells or cells_file or before_id or after_id or chapter or replace:
            cli_error("Literal replacement mode only accepts path, old_str, new_str, run_test, validate_code, dry_run, and show_cells")
        return _write_literal_replacements(path, old_str, new_str, run_test=run_test, validate_code=validate_code, dry_run=dry_run, show_cells=show_cells)
    if show_cells: cli_error("show_cells is only supported with old_str/new_str literal replacement mode")
    if dry_run: cli_error("dry_run is only supported with old_str/new_str literal replacement mode")
    if before_id and after_id: cli_error("Use either before_id or after_id, not both")
    if (before_id or after_id) and chapter is not None: cli_error("Use id-based insertion or chapter insertion, not both")
    if (before_id or after_id) and replace: cli_error("Use id-based insertion or replace, not both")
    path = Path(path)
    cells = load_cells_text(cells, cells_file)
    new_cells = parse_cells(cells, cell_type)
    if validate_code: validate_code_cells(new_cells)

    with notebook_locks(path):
        if replace and chapter is None:
            nb = new_nb(new_cells)
        else:
            nb = _read_nb(path) if path.exists() else new_nb([])
            if chapter is not None:
                span = one_chapter(nb.cells, chapter, create=True)
                if replace:
                    del nb.cells[span["start"] + 1:span["end"]]
                    target = span["start"] + 1
                else:
                    target = span["end"]
            elif before_id or after_id:
                idx, _ = find_cell_by_id(nb.cells, before_id or after_id)
                target = idx if before_id else idx + 1
            else:
                target = len(nb.cells)
            for offset, cell in enumerate(new_cells):
                nb.cells.insert(target + offset, cell)

        stamp_notebook_metadata(nb)
        _write_nb(nb, path)
        exported = _export_notebook(nb, path) is not None
        msg = f"Wrote {len(nb.cells)} cells to {path}"
        if replace: msg += " using replace"
        if chapter is not None: msg += f" in chapter {chapter!r}"
        if before_id: msg += f" before id={before_id}"
        if after_id: msg += f" after id={after_id}"
        if exported: msg += " and exported with nbdev"
        print(msg)
        if run_test: run_notebook_test(path)
        if run_style:
            print(f"Running chstyle on {path}")
            style_check(path, strict=style_strict)
    return cli_return(path)

In [ ]:
path = demo_path("02_write_update.ipynb")
try:
    write_nb(str(path), "%%code\nvalue = 1\nvalue = value + 1", replace=True)
    cell = _read_nb(path).cells[0]
    assert cell.metadata["nbskill"]["cell_type"] == "code"
    before_text = path.read_text(encoding="utf-8")
    out = _StringIO()
    with _redirect_stdout(out):
        update_cell(str(path), "value = 2", cell_id=cell.id, line_range="2", dry_run=True)
    preview = out.getvalue()
    assert "Dry run: would update lines 2" in preview
    assert "-value = value + 1" in preview and "+value = 2" in preview
    assert path.read_text(encoding="utf-8") == before_text
    update_cell(str(path), "value = 2", cell_id=cell.id, line_range="2")
    assert _read_nb(path).cells[0].source == "value = 1\nvalue = 2"
    assert "source_hash" not in _read_nb(path).cells[0].metadata["nbskill"]
    update_cell(str(path), "", cell_id=cell.id, line_range="1")
    assert _read_nb(path).cells[0].source == "value = 2"
    multi_def = "def first():\n    return 1\n\n\ndef second():\n    return first() + 1"
    update_cell(str(path), multi_def, cell_id=cell.id)
    assert _read_nb(path).cells[0].source == multi_def
finally:
    remove_demo_path(path)

In [ ]:
root = demo_path("02_write_literal")
try:
    root.mkdir()
    one = root / "one.ipynb"
    two = root / "two.ipynb"
    write_nb(str(one), "%%code\ndef old_name():\n    return 1\n---\n%%markdown\nold_name docs", replace=True)
    write_nb(str(two), "%%code\nvalue = old_name()", replace=True)
    out = _StringIO()
    with _redirect_stdout(out):
        write_nb(str(root), old_str="old_name", new_str="new_name", dry_run=True, show_cells=True)
    preview = out.getvalue()
    assert "Dry run: would replace" in preview
    assert "Changed cells:" in preview
    assert "id=" in preview and "matches=" in preview
    assert "-def old_name" in preview and "+def new_name" in preview
    assert "old_name" in _read_nb(one).cells[0].source
    write_nb(str(root), old_str="old_name", new_str="new_name")
    assert "new_name" in _read_nb(one).cells[0].source
    assert "new_name docs" in _read_nb(one).cells[1].source
    assert "new_name" in _read_nb(two).cells[0].source
    assert _read_nb(two).cells[0].metadata["nbskill"]["cell_type"] == "code"
finally:
    remove_demo_path(root)

Wrote 2 cells to nbs/data/02_write_literal/one.ipynb using replace
Wrote 1 cells to nbs/data/02_write_literal/two.ipynb using replace
Replaced 3 matches in 3 cells across 2 notebook(s) (nbs/data/02_write_literal/one.ipynb: 2 matches/2 cells; nbs/data/02_write_literal/two.ipynb: 1 matches/1 cells)


In [ ]:
#| export
def _save_nb(nb, path):
    with notebook_locks(path):
        stamp_notebook_metadata(nb)
        _write_nb(nb, path)
        _export_notebook(nb, path)

### Updating one cell

`update_cell` is the surgical tool. It keeps the original cell id, can replace a whole cell or only a line range, clears stale outputs,.

For whole-cell replacement, pass exactly one notebook cell block: optional `%%code`, `%%markdown`, or `%%raw` marker followed by the cell source. Do not include standalone `---` separators; those mean multiple cells and belong with `write_nb` or `batch_edit_nb`. Use `line_range` or `old_str` when replacing only part of a cell.

In [ ]:
#| export
def _parse_line_range(line_range, n_lines):
    if line_range is None: return None
    value = str(line_range).strip()
    if not value: return None
    if ":" in value:
        start_s, end_s = value.split(":", 1)
        start = int(start_s) if start_s else 1
        end = int(end_s) if end_s else n_lines
    else:
        start = end = int(value)
    if start < 1 or end < start or end > n_lines:
        cli_error(f"line_range must be 1-based and within 1:{n_lines}; got {line_range!r}")
    return start - 1, end


def _replace_line_range(source, line_range, new):
    lines = source.splitlines()
    start, end = _parse_line_range(line_range, len(lines) or 1)
    replacement = [] if new == "" else new.splitlines()
    return "\n".join([*lines[:start], *replacement, *lines[end:]])


@call_parse
@tracked_call
def update_cell(
    path: str,  # Notebook path
    new: Param("Replacement cell source, replacement text, or line-range replacement", str, opt=False, nargs="?") = "",
    new_file: str | None = None,  # Read replacement text from a UTF-8 file
    cell_id: str | None = None,  # Stable notebook cell id to update
    old_str: str | None = None,  # Text to replace, or text used to find the target cell
    line_range: str | None = None,  # 1-based inclusive lines to replace, e.g. 3 or 3:5
    cell_type: str = "code",  # Default type for whole-cell replacements without %% marker
    run_test: bool = False,  # Execute the notebook with execnb after writing
    validate_code: bool = True,  # Validate changed Python code before writing
    dry_run: bool = False,  # Show the update plan without writing
):
    "Update one notebook cell by id, replace old_str, or replace a 1-based line range."
    if cell_id is None and old_str is None: cli_error("Pass --cell_id, --old_str, or both")
    if line_range is not None and cell_id is None: cli_error("Pass --cell_id with --line_range")
    path = Path(path)
    new = load_cells_text(new, new_file)

    with notebook_locks(path):
        nb = _read_nb(path)
        idx, cell = find_cell_by_id(nb.cells, cell_id) if cell_id else find_cell_by_text(nb.cells, old_str)
        if old_str is not None and old_str not in cell_source(cell):
            cli_error(f"old_str was not found in id={cell.id}")

        if line_range is not None:
            replacement = _replace_line_range(cell_source(cell), line_range, new)
            if validate_code and getattr(cell, "cell_type", None) == "code": validate_code_cells([mk_cell(replacement)])
            mode = f"lines {line_range}"
            if not dry_run:
                cell.source = replacement
                clear_outputs(cell)
        elif old_str is None:
            new_cell = parse_one_cell(new, cell_type)
            if validate_code: validate_code_cells([new_cell])
            clear_outputs(new_cell)
            if not dry_run: replace_cell(nb, idx, new_cell)
            replacement = cell_source(new_cell)
            mode = "cell"
        else:
            replacement = cell_source(cell).replace(old_str, new, 1)
            if validate_code and getattr(cell, "cell_type", None) == "code": validate_code_cells([mk_cell(replacement)])
            mode = "text"
            if not dry_run:
                cell.source = replacement
                clear_outputs(cell)

        msg = f"{'Dry run: would update' if dry_run else 'Updated'} {mode} id={cell.id}"
        if dry_run:
            diff = _literal_cell_diff(cell_source(cell), replacement)
            if diff: msg += chr(10) + diff
            print(msg)
            return cli_return(path)
        stamp_notebook_metadata(nb)
        _write_nb(nb, path)
        if _export_notebook(nb, path) is not None:
            msg += " and exported with nbdev"
        print(msg)
        if run_test: run_notebook_test(path)
    return cli_return(path)

Batch editing

Agents often need to change several notebook cells together. `batch_edit_nb` accepts a JSON edit plan, runs the same notebook-aware validation and locking as the single-cell tools, and prints a compact diff before writing. Use it when repeated shell calls would make multiline code or dry-run diffs fragile.

In [ ]:
#| export
def _load_batch_plan(plan="", plan_file=None):
    text = load_cells_text(plan, plan_file)
    if not str(text).strip(): cli_error("Pass a JSON batch edit plan or --plan_file")
    try: data = json.loads(text)
    except json.JSONDecodeError as exc: cli_error(f"Batch edit plan must be JSON: {exc}")
    if isinstance(data, list): data = {"operations": data}
    if not isinstance(data, dict) or not isinstance(data.get("operations"), list):
        cli_error("Batch edit plan must be a JSON object with an operations list")
    return data

In [ ]:
#| export
def _op_path(op, default_path=None):
    path = op.get("path") or default_path
    if not path: cli_error(f"Batch operation missing path: {op}")
    return Path(path)

In [ ]:
#| export
def _op_source(op):
    for key in ("source", "new", "text"):
        if key in op: return str(op[key])
    cli_error(f"Batch operation missing source/new/text: {op}")

In [ ]:
#| export
def _op_cells(op, default_cell_type="code"):
    if "cells" in op: return parse_cells(str(op["cells"]), op.get("cell_type", default_cell_type))
    return [parse_one_cell(_op_source(op), op.get("cell_type", default_cell_type))]

In [ ]:
#| export
def _op_diff(before, after, limit=24):
    lines = list(difflib.unified_diff(
        before.splitlines(), after.splitlines(), fromfile="before", tofile="after", lineterm="", n=2,
    ))
    if len(lines) > limit: lines = [*lines[:limit], "... diff truncated ..."]
    return "\n".join(lines)

In [ ]:
#| export
def _batch_detail(op, path, cell_id="", before="", after=""):
    detail = {
        "op": op.get("op"),
        "path": str(path),
        "cell_id": cell_id,
        "diff": _op_diff(before, after) if before != after else "",
    }
    return detail

In [ ]:
#| export
def _apply_batch_op(nb, path, op, validate_code=True, default_cell_type="code"):
    kind = op.get("op")
    if kind in {"set_cell_source", "set_cell"}:
        _, cell = find_cell_by_id(nb.cells, op.get("cell_id"))
        before = cell_source(cell)
        source = _op_source(op)
        cell_type = op.get("cell_type")
        if cell_type: cell.cell_type = cell_type
        if validate_code and getattr(cell, "cell_type", None) == "code": validate_code_cells([mk_cell(source)])
        cell.source = source
        clear_outputs(cell)
        return [_batch_detail(op, path, cell.id, before, source)]
    if kind in {"insert_after_id", "insert_before_id"}:
        idx, anchor = find_cell_by_id(nb.cells, op.get("cell_id") or op.get("after_id") or op.get("before_id"))
        new_cells = _op_cells(op, default_cell_type=default_cell_type)
        if validate_code: validate_code_cells(new_cells)
        target = idx + 1 if kind == "insert_after_id" else idx
        for offset, cell in enumerate(new_cells):
            nb.cells.insert(target + offset, cell)
        where = "after" if kind == "insert_after_id" else "before"
        return [{
            "op": kind,
            "path": str(path),
            "cell_id": getattr(anchor, "id", ""),
            "diff": f"inserted {len(new_cells)} cell(s) {where} id={getattr(anchor, 'id', '')}",
        }]
    if kind == "delete_cell_id":
        idx, cell = find_cell_by_id(nb.cells, op.get("cell_id"))
        before = cell_source(cell)
        del nb.cells[idx]
        return [_batch_detail(op, path, cell.id, before, "")]
    if kind == "replace_text":
        old = op.get("old_str", op.get("old"))
        new = op.get("new_str", op.get("new"))
        if old in {None, ""}: cli_error(f"replace_text needs old/old_str: {op}")
        if new is None: cli_error(f"replace_text needs new/new_str: {op}")
        _, matches, details = _replace_literal_in_notebook(nb, str(old), str(new), validate_code=validate_code, collect_details=True)
        if not matches: cli_error(f"No matches for {old!r} in {path}")
        return [
            {"op": kind, "path": str(path), "cell_id": item["cell_id"], "diff": item["diff"]}
            for item in details
        ]
    cli_error(f"Unknown batch operation {kind!r}")

In [ ]:
#| export
def _format_batch_details(details):
    lines = []
    for item in details:
        cell = f" id={item['cell_id']}" if item.get("cell_id") else ""
        lines.append(f"- {item['op']}: {item['path']}{cell}")
        if item.get("diff"):
            lines.extend(f"    {line}" for line in item["diff"].splitlines())
    return "\n".join(lines)

In [ ]:
#| export
@call_parse
@tracked_call
def batch_edit_nb(
    plan: Param("JSON edit plan, or - to read stdin", str, opt=False, nargs="?") = "",
    plan_file: str | None = None,  # Read the JSON plan from a UTF-8 file
    path: str | None = None,  # Default notebook path for operations that omit path
    dry_run: bool = True,  # Show the plan and diffs without writing
    validate_code: bool = True,  # Validate changed Python code before writing
    default_cell_type: str = "code",  # Default cell type for inserted cells without %% markers
):
    "Apply a JSON batch edit plan to one or more notebooks with locks and diffs."
    data = _load_batch_plan(plan, plan_file)
    ops = data["operations"]
    paths = sorted({_op_path(op, path) for op in ops}, key=str)
    details = []
    exported = False
    with notebook_locks(*paths):
        notebooks = {nb_path: _read_nb(nb_path) for nb_path in paths}
        for op in ops:
            nb_path = _op_path(op, path)
            details.extend(_apply_batch_op(notebooks[nb_path], nb_path, op, validate_code=validate_code, default_cell_type=default_cell_type))
        if not dry_run:
            for nb_path, nb in notebooks.items():
                stamp_notebook_metadata(nb)
                _write_nb(nb, nb_path)
                exported = _export_notebook(nb, nb_path) is not None or exported
    prefix = "Dry run: would apply" if dry_run else "Applied"
    msg = f"{prefix} {len(ops)} batch operations across {len(paths)} notebook(s)"
    if exported: msg += " and exported with nbdev"
    if details: msg += "\n" + _format_batch_details(details)
    print(msg)
    return cli_return([str(path) for path in paths])

### Splitting one chapter

`split_nb_chapter` moves one `##` chapter into a new nbdev notebook. It copies imports used by the moved code, imports source-notebook definitions that the moved chapter still references, and promotes referenced private source helpers by dropping the leading underscore. It is intentionally CLI-only, because splitting modules is a broad refactor that should be run deliberately from the shell.

In [ ]:
#| export
def _cell_lines(cell):
    source = cell_source(cell)
    return source.splitlines()


def _default_exp_from_cells(cells):
    for cell in cells:
        for line in _cell_lines(cell):
            match = re.match(r"^\s*#\|\s*default_exp\s+(.+?)\s*$", line)
            if match: return match.group(1).strip()
    return None


def _default_exp_for_dest(dest):
    dest = Path(dest)
    try:
        from nbdev.config import get_config
        cfg = get_config()
        nbs_path = Path(cfg.config_path) / cfg.nbs_path
        rel = dest.resolve().relative_to(nbs_path.resolve()).with_suffix("")
        return ".".join(rel.parts)
    except Exception:
        return dest.stem


def _module_for_default_exp(default_exp, path=None):
    if not default_exp: return None
    try:
        from nbdev.config import get_config
        cfg = get_config(Path(path).parent if path else None)
        lib_name = str(cfg.lib_name)
    except Exception:
        lib_name = None
    if lib_name and not str(default_exp).startswith(f"{lib_name}."):
        return f"{lib_name}.{default_exp}"
    return str(default_exp)


def _is_default_exp_cell(cell):
    return any(re.match(r"^\s*#\|\s*default_exp\s+", line) for line in _cell_lines(cell))


def _code_ast(cell):
    if getattr(cell, "cell_type", None) != "code": return None
    try: return ast.parse(cell_source(cell))
    except SyntaxError: return None


def _import_bound_names(node):
    if isinstance(node, ast.Import):
        return [alias.asname or alias.name.split(".", 1)[0] for alias in node.names]
    if isinstance(node, ast.ImportFrom):
        return [alias.asname or alias.name for alias in node.names if alias.name != "*"]
    return []


def _node_source_from_cell(cell, node):
    lines = cell_source(cell).splitlines()
    start = min([node.lineno, *[d.lineno for d in getattr(node, "decorator_list", [])]]) - 1
    return "\n".join(lines[start:node.end_lineno]).strip("\n")


def _definition_names(cells):
    names = {}
    for idx, cell in enumerate(cells):
        tree = _code_ast(cell)
        if tree is None: continue
        for node in tree.body:
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                names[node.name] = idx
    return names


def _bound_names(cells):
    names = set()
    for cell in cells:
        tree = _code_ast(cell)
        if tree is None: continue
        for node in tree.body:
            names.update(_import_bound_names(node))
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                names.add(node.name)
        for node in ast.walk(tree):
            if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Store): names.add(node.id)
    return names


def _loaded_names(cells):
    names = set()
    for cell in cells:
        tree = _code_ast(cell)
        if tree is None: continue
        for node in ast.walk(tree):
            if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Load): names.add(node.id)
    return names - _bound_names(cells) - set(dir(builtins))


def _import_lines_for_names(cells, names):
    lines, seen = [], set()
    for cell in cells:
        tree = _code_ast(cell)
        if tree is None: continue
        for node in tree.body:
            bound = set(_import_bound_names(node))
            if not bound or not (bound & names): continue
            line = ast.unparse(node)
            if line not in seen:
                seen.add(line)
                lines.append(line)
    return lines


def _symbol_replacements(promotions):
    return {old: new for old, new in promotions}


def _replace_symbol_refs(source, promotions):
    for old, new in _symbol_replacements(promotions).items():
        source = re.sub(rf"(?<![\w.]){re.escape(old)}(?![\w])", new, source)
    return source


def _apply_promotions(cells, promotions):
    if not promotions: return
    for cell in cells:
        if getattr(cell, "cell_type", None) == "code":
            cell.source = _replace_symbol_refs(cell_source(cell), promotions)
            clear_outputs(cell)


def _insert_import_cell(cells, lines):
    lines = [line for line in lines if line]
    if not lines: return
    source = "#| export\n" + "\n".join(dict.fromkeys(lines))
    insert_at = 1 if cells and _is_default_exp_cell(cells[0]) else 0
    cells.insert(insert_at, mk_cell(source, cell_type="code"))


def _split_chapter_plan(nb, chapter, dest, default_exp=None, promote_private=True):
    span = one_chapter(nb.cells, chapter)
    moved = [copy.deepcopy(cell) for cell in nb.cells[span["start"]:span["end"]]]
    remaining = [copy.deepcopy(cell) for idx, cell in enumerate(nb.cells) if not (span["start"] <= idx < span["end"])]
    moved_uses = _loaded_names(moved)
    remaining_uses = _loaded_names(remaining)
    outside_defs = _definition_names(remaining)
    moved_defs = _definition_names(moved)
    source_default_exp = _default_exp_from_cells(nb.cells)
    source_module = _module_for_default_exp(source_default_exp, path=dest)
    dest_default_exp = default_exp or _default_exp_for_dest(dest)
    dest_module = _module_for_default_exp(dest_default_exp, path=dest)

    source_deps = sorted(moved_uses & set(outside_defs))
    moved_deps = sorted(remaining_uses & set(moved_defs))
    promotions = []
    for name in source_deps:
        if name.startswith("_") and promote_private:
            public = name.lstrip("_")
            if public in outside_defs or public in moved_uses:
                cli_error(f"Cannot promote {name!r}: {public!r} already exists or is referenced")
            promotions.append((name, public))
    _apply_promotions(remaining, promotions)

    import_lines = _import_lines_for_names(remaining, moved_uses - set(outside_defs))
    if source_deps:
        if not source_module: cli_error("Source notebook needs #| default_exp before split dependencies can be imported")
        for name in source_deps:
            promoted = dict(promotions).get(name, name)
            import_lines.append(f"from {source_module} import {promoted} as {name}" if promoted != name else f"from {source_module} import {name}")
    dest_cells = [mk_cell(f"#| default_exp {dest_default_exp}", cell_type="code")]
    _insert_import_cell(dest_cells, import_lines)
    dest_cells.extend(moved)

    source_imports = []
    if moved_deps:
        if not dest_module: cli_error("Destination notebook needs #| default_exp before source dependencies can be imported")
        for name in moved_deps:
            source_imports.append(f"from {dest_module} import {name}")
    _insert_import_cell(remaining, source_imports)

    return {
        "span": span,
        "source_default_exp": source_default_exp,
        "dest_default_exp": dest_default_exp,
        "source_module": source_module,
        "dest_module": dest_module,
        "source_dependencies": source_deps,
        "moved_dependencies": moved_deps,
        "copied_imports": import_lines,
        "source_imports": source_imports,
        "promotions": promotions,
        "source_cells": remaining,
        "dest_cells": dest_cells,
    }


def _format_split_plan(path, dest, chapter, plan, dry_run=False):
    prefix = "Dry run: would split" if dry_run else "Split"
    lines = [f"{prefix} chapter {chapter!r} from {path} -> {dest}"]
    lines.append(f"moved cells={len(plan['dest_cells']) - 1}; destination default_exp={plan['dest_default_exp']}")
    if plan["copied_imports"]:
        lines.append("destination imports:")
        lines.extend(f"- {line}" for line in plan["copied_imports"])
    if plan["source_imports"]:
        lines.append("source imports:")
        lines.extend(f"- {line}" for line in plan["source_imports"])
    if plan["promotions"]:
        lines.append("promoted source helpers:")
        lines.extend(f"- {old} -> {new}" for old, new in plan["promotions"])
    return "\n".join(lines)


@call_parse
@tracked_call
def split_nb_chapter(
    path: str,  # Source notebook path
    chapter: str,  # Chapter title string or regex to split out
    dest: str,  # Destination notebook path
    default_exp: str | None = None,  # Destination nbdev default_exp; defaults from dest path
    dry_run: bool = True,  # Show the split plan without writing notebooks
    force: bool = False,  # Overwrite dest if it already exists
    promote_private: bool = True,  # Promote referenced private source helpers by dropping the leading underscore
):
    "Split one ## chapter into a new nbdev notebook."
    path, dest = Path(path), Path(dest)
    if dest.exists() and not force: cli_error(f"Destination exists: {dest}; pass --force to overwrite")
    with notebook_locks(path, dest):
        nb = _read_nb(path)
        plan = _split_chapter_plan(nb, chapter=chapter, dest=dest, default_exp=default_exp, promote_private=promote_private)
        msg = _format_split_plan(path, dest, chapter, plan, dry_run=dry_run)
        print(msg)
        if dry_run: return cli_return(plan)
        source_nb = new_nb(plan["source_cells"])
        dest_nb = new_nb(plan["dest_cells"])
        validate_code_cells([cell for cell in source_nb.cells if getattr(cell, "cell_type", None) == "code"])
        validate_code_cells([cell for cell in dest_nb.cells if getattr(cell, "cell_type", None) == "code"])
        stamp_notebook_metadata(source_nb)
        stamp_notebook_metadata(dest_nb)
        dest.parent.mkdir(parents=True, exist_ok=True)
        _write_nb(source_nb, path)
        _write_nb(dest_nb, dest)
        _export_notebook(source_nb, path)
        _export_notebook(dest_nb, dest)
    return cli_return(plan)


In [ ]:
plan_path = demo_path("batch_edit_plan.json")
nb_path = demo_path("batch_edit_target.ipynb")
try:
    write_nb(str(nb_path), "%%code\nvalue = 1", replace=True)
    nb = _read_nb(nb_path)
    cell = nb.cells[0]
    plan = {
        "operations": [
            {"op": "set_cell_source", "path": str(nb_path), "cell_id": cell.id, "source": "value = 2"},
            {"op": "insert_after_id", "path": str(nb_path), "cell_id": cell.id, "cells": "%%code\nassert value == 2"},
        ]
    }
    plan_path.write_text(json.dumps(plan), encoding="utf-8")
    batch_edit_nb(plan_file=str(plan_path), dry_run=True)
    batch_edit_nb(plan_file=str(plan_path), dry_run=False)
    nb = _read_nb(nb_path)
    assert cell_source(nb.cells[0]) == "value = 2"
    assert "assert value == 2" in cell_source(nb.cells[1])
finally:
    remove_demo_path(plan_path)
    remove_demo_path(nb_path)

Wrote 1 cells to nbs/data/batch_edit_target.ipynb using replace
Dry run: would apply 2 batch operations across 1 notebook(s)
- set_cell_source: nbs/data/batch_edit_target.ipynb id=5bec1490
    --- before
    +++ after
    @@ -1 +1 @@
    -value = 1
    +value = 2
- insert_after_id: nbs/data/batch_edit_target.ipynb id=5bec1490
    inserted 1 cell(s) after id=5bec1490
Applied 2 batch operations across 1 notebook(s)
- set_cell_source: nbs/data/batch_edit_target.ipynb id=5bec1490
    --- before
    +++ after
    @@ -1 +1 @@
    -value = 1
    +value = 2
- insert_after_id: nbs/data/batch_edit_target.ipynb id=5bec1490
    inserted 1 cell(s) after id=5bec1490
